# PCN Detection — Method Comparison & Performance Benchmark

Compares the proposed detector (**regex + Luhn**) against a **regex-only baseline** on two axes:

1. **Accuracy** on the labelled evaluation corpus — accuracy, precision, recall, F1 (field-level and span-level)
2. **Edge cases** — a hand-built suite of tricky inputs, each marked PASS / LEAK / OVER-MASK
3. **Performance at scale** — runtime, throughput **and accuracy** as dataset **rows** and **columns** grow, plus the real Faker tables

Both methods use the pipeline's own code: the regex + Luhn logic from `pcn_utils.py` and the Pandas UDF from `pcn_spark.py`, exactly as `pcn_detection_masking.ipynb` imports them. The only difference between the two methods is the Luhn check.

**Prerequisites:** run `generate_data.ipynb` and `generate_evaluation_corpus.ipynb` first.

**Outputs (Delta, appended per run with a `run_id`):** `pcn_method_comparison`, `pcn_edge_case_results`, `pcn_benchmark_results`.

---
## Setup

In [0]:
import random
import re
import statistics
import time
import uuid
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import StrMethodFormatter
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

from pcn_eval import TEXT_COLUMNS, compute_prf, evaluate_corpus, summarise_metrics
from pcn_spark import make_mask_udf
from pcn_utils import (
    SEPARATORS,
    generate_card_number, generate_luhn_failing_number,
    mask_pcns_in_string, mask_pcns_regex_only,
)

EVAL_DATABASE = "capstone_project.pii_data"
EVAL_TABLE = "pcn_evaluation_corpus"
COMPARISON_TABLE = "pcn_method_comparison"
EDGE_CASE_TABLE = "pcn_edge_case_results"
BENCHMARK_TABLE = "pcn_benchmark_results"
BENCH_TABLE_PREFIX = "pcn_bench_input"

# Same delimiter and NULL sentinel as pcn_detection_masking.ipynb
CONCAT_DELIMITER = "||"
NULL_SENTINEL = "__NULL__"
SPLIT_PATTERN = re.escape(CONCAT_DELIMITER)

# --- Benchmark grid (shrink these if Free Edition runs are too slow) ---
ROW_SIZES = [10_000, 100_000, 500_000, 1_000_000]  # row sweep, at ROW_SWEEP_COLS columns
ROW_SWEEP_COLS = 8
COLUMN_COUNTS = [4, 8, 16, 32, 64]                 # column sweep, at COL_SWEEP_ROWS rows
COL_SWEEP_ROWS = 100_000
PCN_CELL_RATE = 0.05        # share of cells holding a Luhn-valid PCN
HARD_NEG_CELL_RATE = 0.05   # share of cells holding a 13–19 digit Luhn-failing number
MERGED_PCN_CELL_RATE = 0.0  # optional: PCN followed by a CVV ("… 4111 1111 1111 1111 123"), a known Luhn leak case
REPEATS = 3                 # timed runs per method per dataset, after one warm-up
REBUILD_INPUTS = False      # True = regenerate synthetic input tables even if they exist
SHAPE_TABLES = ["credit_card_transactions", "customer_profiles", "non_credit_transactions", EVAL_TABLE]

# Ground truth for accuracy at scale
LABEL_COL = "_pcn_labels"  # synthetic tables: comma-separated 1/0 per column ("1" = PCN); never masked
PCN_COLUMNS = {            # Faker tables: columns that always hold a PCN (see generate_data.ipynb)
    "credit_card_transactions": ["transaction_format", "card_number"],
    "customer_profiles": ["credit_card_number"],
    "non_credit_transactions": [],
}

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().isoformat()

METHOD_LABELS = {"regex_only": "Regex only (baseline)", "regex_luhn": "Regex + Luhn"}
METHOD_COLORS = {"regex_only": "#eb6834", "regex_luhn": "#2a78d6"}
# Line style + marker as a second cue, so overlapping lines (e.g. both at recall 1.0) stay visible
METHOD_STYLES = {"regex_only": ("-", "o"), "regex_luhn": ("--", "s")}

print(f"Run ID: {RUN_ID}")
print(f"Spark version: {spark.version}")

---
## Methods Under Test

| Method | Candidate detection | Validation | Masked when |
|---|---|---|---|
| `regex_only` (baseline) | `(?<!\d)(?:\d[\s-]*?){12,18}\d(?!\d)` | none | every regex match with 13–19 digits |
| `regex_luhn` (proposed) | same regex | Luhn mod-10 | regex match with 13–19 digits **and** passes Luhn |

Both run through `make_mask_udf` from `pcn_spark.py`, the same Pandas UDF the pipeline uses (vectorised `str.contains` pre-filter, then row-level masking on candidate rows only). `regex_luhn` is `make_mask_udf(mask_pcns_in_string)`, identical to `mask_pcns` in `pcn_detection_masking.ipynb`; `regex_only` swaps in `mask_pcns_regex_only`, which is the same function with the Luhn check switched off.

In [0]:
# regex_luhn is built exactly as mask_pcns is in pcn_detection_masking.ipynb
METHODS = {
    "regex_only": make_mask_udf(mask_pcns_regex_only),
    "regex_luhn": make_mask_udf(mask_pcns_in_string),
}
print(f"Methods: {list(METHODS)}")

---
## Part 1 — Accuracy on the Evaluation Corpus

**Definitions** (same scoring as `evaluation_harness.ipynb`, shared via `pcn_eval.py`):

| Metric | Formula | Meaning here |
|---|---|---|
| Accuracy | (TP + TN) / all | share of fields classified correctly (field-level only — span scoring has no true negatives) |
| Precision | TP / (TP + FP) | of the things we masked, how many were real PCNs |
| Recall | TP / (TP + FN) | of the real PCNs, how many we masked (the PCI DSS-critical number) |
| F1 | 2PR / (P + R) | balance of precision and recall |

- **Field-level:** did we mask *something* in each field that contains a PCN, and nothing in fields that don't?
- **Span-level:** was every character of each annotated PCN masked, and were any unannotated characters masked?

In [0]:
corpus_sdf = spark.table(f"{EVAL_DATABASE}.{EVAL_TABLE}")

# Masking runs on the workers; one masked column per (method, text column)
masked_sdf = corpus_sdf.select(
    "*",
    *[METHODS[m](F.col(c)).alias(f"{c}__{m}") for m in METHODS for c in TEXT_COLUMNS],
)

# The corpus is a few hundred rows, so scoring it on the driver is cheap
corpus_df = masked_sdf.toPandas()
print(f"Loaded {len(corpus_df)} corpus rows")
print(f"Categories: {corpus_df['test_category'].value_counts().to_dict()}")

In [0]:
metrics, span_frames, field_frames, fp_frames = {}, {}, {}, {}
for method in METHODS:
    span_df, field_df, fp_df = evaluate_corpus(corpus_df, TEXT_COLUMNS, masked_suffix=f"__{method}")
    metrics[method] = summarise_metrics(span_df, field_df)
    span_frames[method], field_frames[method], fp_frames[method] = span_df, field_df, fp_df

comparison_df = pd.DataFrame(metrics)
comparison_df["luhn_minus_baseline"] = comparison_df["regex_luhn"] - comparison_df["regex_only"]

RATE_METRICS = [
    "field_accuracy", "field_precision", "field_recall", "field_f1",
    "span_precision", "span_recall", "span_f1",
]
COUNT_METRICS = [
    "field_tp", "field_fp", "field_fn", "field_tn",
    "span_tp", "span_fp", "span_fn",
]

print("RATES")
display(comparison_df.loc[RATE_METRICS].round(4))
print("COUNTS")
display(comparison_df.loc[COUNT_METRICS].astype(int))

In [0]:
fig, ax = plt.subplots(figsize=(10, 4))
bar_width = 0.38
positions = range(len(RATE_METRICS))

for offset, method in zip((-bar_width / 2, bar_width / 2), METHODS):
    values = [metrics[method][m] for m in RATE_METRICS]
    bars = ax.bar(
        [p + offset for p in positions], values, width=bar_width - 0.02,
        color=METHOD_COLORS[method], label=METHOD_LABELS[method],
    )
    ax.bar_label(bars, fmt="%.3f", fontsize=8, color="#52514e", padding=2)

ax.set_xticks(list(positions))
ax.set_xticklabels([m.replace("_", "\n") for m in RATE_METRICS], color="#52514e")
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score", color="#52514e")
ax.set_title("Detection accuracy: regex + Luhn vs regex-only baseline", loc="left", fontsize=11)
ax.grid(axis="y", color="#e4e3df", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

### Where the methods disagree

Field-level results per test label, showing only the labels where the two methods score differently.

In [0]:
def per_label_summary(field_df):
    counts = (
        field_df.groupby(["test_category", "test_label"])["verdict"]
        .value_counts().unstack(fill_value=0)
        .reindex(columns=["TP", "FP", "FN", "TN"], fill_value=0)
    )
    counts["accuracy"] = (counts["TP"] + counts["TN"]) / counts.sum(axis=1)
    return counts


by_label = pd.concat({m: per_label_summary(field_frames[m]) for m in METHODS}, axis=1)
differs = by_label[("regex_only", "accuracy")] != by_label[("regex_luhn", "accuracy")]
print(f"{int(differs.sum())} of {len(by_label)} test labels score differently between the methods")
display(by_label[differs].round(3))

for method in METHODS:
    fp_df = fp_frames[method]
    print(f"\n{METHOD_LABELS[method]}: {len(fp_df)} false-positive span(s)")
    if len(fp_df):
        display(fp_df.head(10))

In [0]:
comparison_rows = [
    {
        "run_id": RUN_ID,
        "run_timestamp": RUN_TIMESTAMP,
        "method": method,
        "corpus_size": len(corpus_df),
        **metrics[method],
    }
    for method in METHODS
]
(
    spark.createDataFrame(pd.DataFrame(comparison_rows))
    .write.format("delta").mode("append").option("mergeSchema", "true")
    .saveAsTable(f"{EVAL_DATABASE}.{COMPARISON_TABLE}")
)
print(f"✔ Accuracy comparison appended to '{EVAL_DATABASE}.{COMPARISON_TABLE}'.")

---
## Part 2 — Edge Cases

Targeted inputs the corpus doesn't cover. Each case lists the substrings that are genuine PCNs; the verdict per method is:

- **PASS** — every PCN fully masked, nothing else touched
- **LEAK** — at least one PCN character left visible (compliance failure)
- **OVER-MASK** — non-PCN characters masked (data-quality cost, not a compliance failure)

The *concatenation* group mirrors the pipeline, which joins all columns with `||` before masking: a PCN must be masked without disturbing the delimiters, and digits in neighbouring columns must never merge into one PCN.

In [0]:
# (case_id, group, text, substrings that are genuine PCNs and must be masked)
EDGE_CASES = [
    # Column concatenation ('||' joins columns in the pipeline)
    ("delim_card_between_columns", "concatenation", "amy||4111111111111111||x", ["4111111111111111"]),
    ("delim_cards_in_adjacent_columns", "concatenation", "4111 1111 1111 1111||5500 0000 0000 0004",
     ["4111 1111 1111 1111", "5500 0000 0000 0004"]),
    ("delim_digits_split_across_columns", "concatenation", "411111||1111111111", []),
    ("delim_null_sentinels", "concatenation", "__NULL__||4111-1111-1111-1111||__NULL__", ["4111-1111-1111-1111"]),
    ("delim_single_pipe", "concatenation", "4111111111111111|5500000000000004",
     ["4111111111111111", "5500000000000004"]),
    # Separator formats
    ("sep_mixed", "separators", "4111 1111-1111 1111", ["4111 1111-1111 1111"]),
    ("sep_double_space", "separators", "4111  1111  1111  1111", ["4111  1111  1111  1111"]),
    ("sep_spaced_hyphen", "separators", "4111 - 1111 - 1111 - 1111", ["4111 - 1111 - 1111 - 1111"]),
    ("sep_newline", "separators", "4111\n1111\n1111\n1111", ["4111\n1111\n1111\n1111"]),
    ("sep_trailing_hyphen", "separators", "x 4111-1111-1111-1111-", ["4111-1111-1111-1111"]),
    ("sep_dots", "separators", "4111.1111.1111.1111", ["4111.1111.1111.1111"]),
    ("sep_slashes", "separators", "4111/1111/1111/1111", ["4111/1111/1111/1111"]),
    # Surrounding text
    ("text_glued_letters", "surrounding text", "card4111111111111111end", ["4111111111111111"]),
    ("text_fullwidth_digits", "surrounding text", "４１１１１１１１１１１１１１１１", ["４１１１１１１１１１１１１１１１"]),
    ("text_labelled_expiry_cvv", "surrounding text", "4111-1111-1111-1111 exp 12/28 cvv 123", ["4111-1111-1111-1111"]),
    ("text_card_then_4_digits", "surrounding text", "Order 4111111111111111 1234", ["4111111111111111"]),
    # PCN next to another number: regex merges them into one 17–19 digit candidate
    ("merge_card_then_cvv", "merged numbers", "4111 1111 1111 1111 123", ["4111 1111 1111 1111"]),
    ("merge_card_then_expiry", "merged numbers", "4111 1111 1111 1111 12/28", ["4111 1111 1111 1111"]),
    ("merge_qty_then_card", "merged numbers", "Qty 2 4111111111111111", ["4111111111111111"]),
    ("merge_amex_then_pin", "merged numbers", "378282246310005 1234", ["378282246310005"]),
    # Length boundaries
    ("len_13_visa", "length", "4222222222222", ["4222222222222"]),
    ("len_14_diners", "length", "30569309025904", ["30569309025904"]),
    ("len_15_amex_grouped", "length", "3782 822463 10005", ["3782 822463 10005"]),
    ("len_19_maestro", "length", "6759649826438453003", ["6759649826438453003"]),
    ("len_12_luhn_valid", "length", "ref 123456789015", []),
    ("len_20_luhn_valid", "length", "ref 12345678901234567894", []),
    # Hard negatives
    ("neg_luhn_failing_16", "hard negatives", "ref 4111111111111112", []),
    ("neg_all_zeros", "hard negatives", "0000000000000000", []),
    ("neg_compact_timestamp", "hard negatives", "20260927 123045", []),
    ("neg_iso_timestamp", "hard negatives", "2026-09-27 12:30:45", []),
    ("neg_phone", "hard negatives", "+1-415-555-0199", []),
    ("neg_iban", "hard negatives", "GB82 WEST 1234 5698 7654 32", []),
    # Empty values
    ("empty_string", "empty", "", []),
    ("null_value", "empty", None, []),
]
EXPECTED_PCNS = {case_id: pcns for case_id, _, _, pcns in EDGE_CASES}

for case_id, _, text, pcns in EDGE_CASES:
    for pcn in pcns:
        assert pcn in text, f"{case_id}: '{pcn}' not found in its text"
print(f"{len(EDGE_CASES)} edge cases defined.")

In [0]:
def assess_masking(text, masked, pcns):
    # PASS / LEAK / OVER-MASK / LEAK+OVER-MASK for one masked string
    if text is None:
        return "PASS" if masked is None else "OVER-MASK"
    expected = set()
    for pcn in pcns:
        start = text.index(pcn)
        expected.update(range(start, start + len(pcn)))
    changed = {i for i, (a, b) in enumerate(zip(text, masked)) if a != b}
    leak = not expected <= changed
    over = bool(changed - expected)
    if leak and over:
        return "LEAK+OVER-MASK"
    if leak:
        return "LEAK"
    if over:
        return "OVER-MASK"
    return "PASS"


edge_sdf = spark.createDataFrame(
    [(case_id, group, text) for case_id, group, text, _ in EDGE_CASES],
    schema="case_id string, case_group string, text string",
)
edge_df = edge_sdf.select("*", *[METHODS[m](F.col("text")).alias(f"masked_{m}") for m in METHODS]).toPandas()

for method in METHODS:
    edge_df[f"verdict_{method}"] = [
        assess_masking(text, masked, EXPECTED_PCNS[case_id])
        for case_id, text, masked in zip(edge_df["case_id"], edge_df["text"], edge_df[f"masked_{method}"])
    ]

verdict_cols = [f"verdict_{m}" for m in METHODS]
summary = pd.DataFrame({METHOD_LABELS[m]: edge_df[f"verdict_{m}"].value_counts() for m in METHODS}).fillna(0).astype(int)
print("EDGE-CASE VERDICTS")
display(summary)

pd.set_option("display.max_colwidth", 60)
display(edge_df[["case_group", "case_id", "text", *verdict_cols, *[f"masked_{m}" for m in METHODS]]])

In [0]:
(
    spark.createDataFrame(edge_df.assign(run_id=RUN_ID, run_timestamp=RUN_TIMESTAMP))
    .write.format("delta").mode("append").option("mergeSchema", "true")
    .saveAsTable(f"{EVAL_DATABASE}.{EDGE_CASE_TABLE}")
)
print(f"✔ Edge-case results appended to '{EVAL_DATABASE}.{EDGE_CASE_TABLE}'.")

---
## Part 3 — Performance Benchmark

**Method**

- **Synthetic inputs** are generated *on the workers* (`mapInPandas` over `spark.range`), so no row data is built on the driver. Each cell is a PCN transaction string (`PCN_CELL_RATE`), a Luhn-failing 13–19 digit reference (`HARD_NEG_CELL_RATE`), or clean text with dates, amounts and short IDs.
- Each input is **written to Delta before timing**, so generation cost is excluded.
- The timed step matches the pipeline: cast → `concat_ws('||')` → masking UDF, forced over every row by an aggregate (no output write, so storage speed doesn't skew the comparison).
- One **warm-up** run per method (starts Python workers, warms the file cache), then `REPEATS` timed runs **interleaved** between methods; the **median** is reported.
- **Sweeps:** rows (`ROW_SIZES` at `ROW_SWEEP_COLS` columns), columns (`COLUMN_COUNTS` at `COL_SWEEP_ROWS` rows), and the real tables from `generate_data.ipynb` plus the evaluation corpus.
- **Accuracy at scale:** each synthetic row carries a hidden `_pcn_labels` column recording which cells hold a PCN (excluded from masking and timing). The Faker tables are labelled by column (`PCN_COLUMNS`). After timing, one extra untimed run per method splits the masked string back on `||` and scores every cell.

**Throughput metrics:** rows/s, cells/s (rows × columns), and MB/s of concatenated input text (1 character ≈ 1 byte).

In [0]:
FIRST_NAMES = ["ALICE", "BOB", "CAROL", "DAVID", "EVA", "FRANK", "GRACE", "HENRY", "IRIS", "JAMES"]
MERCHANTS = ["Tesco", "Amazon", "Shell", "Starbucks", "Uber", "Netflix", "IKEA", "Boots"]


def make_cell_value():
    # Returns (cell text, is_pcn)
    roll = random.random()
    if roll < PCN_CELL_RATE + MERGED_PCN_CELL_RATE:
        formatted, _, _ = generate_card_number(separator_name=random.choice(list(SEPARATORS)))
        text = f"CC PAYMENT FOR {random.choice(FIRST_NAMES)} {formatted}"
        if roll >= PCN_CELL_RATE:  # merged case: CVV straight after the PCN
            text += f" {random.randint(100, 999)}"
        return text, True
    if roll < PCN_CELL_RATE + MERGED_PCN_CELL_RATE + HARD_NEG_CELL_RATE:
        return f"Reference {generate_luhn_failing_number(random.randint(13, 19))}", False
    return (
        f"{random.choice(MERCHANTS)} {random.randint(1, 999)}.{random.randint(0, 99):02d} "
        f"on 2026-{random.randint(1, 12):02d}-{random.randint(1, 28):02d} ref {random.randint(100000, 999999)}"
    ), False


def build_bench_table(n_rows, n_cols):
    # Generate n_rows x n_cols of synthetic text on the workers and save it to Delta
    table = f"{EVAL_DATABASE}.{BENCH_TABLE_PREFIX}_{n_rows}r_{n_cols}c"
    # Reuse an existing input unless asked not to, or it predates ground-truth labels
    if spark.catalog.tableExists(table) and not REBUILD_INPUTS and LABEL_COL in spark.table(table).columns:
        return table

    col_names = [f"col_{j:03d}" for j in range(n_cols)]
    schema = StructType([StructField(c, StringType()) for c in [*col_names, LABEL_COL]])

    def generate(batches):
        for batch in batches:
            if batch.empty:
                continue
            random.seed(int(batch["id"].iloc[0]))  # reproducible per batch
            rows = [[make_cell_value() for _ in col_names] for _ in range(len(batch))]
            out = pd.DataFrame({c: [row[j][0] for row in rows] for j, c in enumerate(col_names)})
            out[LABEL_COL] = [",".join("1" if is_pcn else "0" for _, is_pcn in row) for row in rows]
            yield out

    (
        spark.range(n_rows).mapInPandas(generate, schema)
        .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(table)
    )
    print(f"  built {table}")
    return table

In [0]:
def concat_columns(df):
    # Cast + NULL sentinel + '||' join, as the pipeline does; the label column is never masked
    data_cols = [c for c in df.columns if c != LABEL_COL]
    concat = F.concat_ws(
        CONCAT_DELIMITER,
        *[F.coalesce(F.col(c).cast("string"), F.lit(NULL_SENTINEL)) for c in data_cols],
    )
    return concat, data_cols


def run_masking(table, method):
    # Concat all columns as the pipeline does, mask, and force the UDF over every row
    df = spark.table(table)
    concat, _ = concat_columns(df)
    masked = df.select(concat.alias("src")).select("src", METHODS[method](F.col("src")).alias("masked"))
    # Unique alias per call so a query result cache can't answer a repeated run
    rows_masked_col = f"rows_masked_{uuid.uuid4().hex[:8]}"
    stats = masked.agg(
        F.count(F.lit(1)).alias("n_rows"),
        F.sum((F.col("src") != F.col("masked")).cast("long")).alias(rows_masked_col),
        F.sum(F.length("src")).alias("n_chars"),
    ).first()
    return stats["n_rows"], stats[rows_masked_col], stats["n_chars"]


def score_table(table, method, pcn_columns=None):
    # Cell-level confusion counts for one method, or None if the table has no ground truth
    df = spark.table(table)
    concat, data_cols = concat_columns(df)
    n_cols = len(data_cols)
    if LABEL_COL in df.columns:
        labels = F.split(F.col(LABEL_COL), ",")
    elif pcn_columns is not None:
        labels = F.array(*[F.lit("1" if c in pcn_columns else "0") for c in data_cols])
    else:
        return None

    rows = df.select(concat.alias("src"), labels.alias("label")).select(
        "label",
        F.split("src", SPLIT_PATTERN).alias("orig"),
        F.split(METHODS[method](F.col("src")), SPLIT_PATTERN).alias("masked"),
    )
    # One row per cell. A row whose masked string no longer splits into n_cols
    # parts has had a '||' delimiter damaged and is counted as misaligned.
    cells = rows.select(
        "label", "masked", F.size("masked").alias("n_masked"),
        F.posexplode("orig").alias("pos", "orig"),
    ).select(
        (F.expr("element_at(label, pos + 1)") == "1").alias("is_pcn"),
        (F.col("orig") != F.expr(f"CASE WHEN n_masked = {n_cols} THEN element_at(masked, pos + 1) END")).alias("was_masked"),
        ((F.col("pos") == 0) & (F.col("n_masked") != n_cols)).alias("misaligned_row"),
    )
    is_pcn, was_masked = F.col("is_pcn"), F.col("was_masked")
    c = cells.agg(
        F.sum((is_pcn & was_masked).cast("long")).alias("tp"),
        F.sum((~is_pcn & was_masked).cast("long")).alias("fp"),
        F.sum((is_pcn & ~was_masked).cast("long")).alias("fn"),
        F.sum((~is_pcn & ~was_masked).cast("long")).alias("tn"),
        F.sum(F.col("misaligned_row").cast("long")).alias("misaligned_rows"),
    ).first()

    tp, fp, fn, tn = (int(c[k] or 0) for k in ("tp", "fp", "fn", "tn"))
    precision, recall, f1 = compute_prf(tp, fp, fn)
    total = tp + fp + fn + tn
    return {
        "cell_tp": tp, "cell_fp": fp, "cell_fn": fn, "cell_tn": tn,
        "accuracy": (tp + tn) / total if total else 0.0,
        "precision": precision, "recall": recall, "f1": f1,
        "misaligned_rows": int(c["misaligned_rows"] or 0),
    }


def benchmark_table(table, dataset, sweep, pcn_columns=None):
    n_cols = len([c for c in spark.table(table).columns if c != LABEL_COL])
    for method in METHODS:
        run_masking(table, method)  # warm-up, not recorded

    timings = {m: [] for m in METHODS}
    last_stats = {}
    for _ in range(REPEATS):
        for method in METHODS:  # interleaved so drift hits both methods equally
            start = time.perf_counter()
            last_stats[method] = run_masking(table, method)
            timings[method].append(time.perf_counter() - start)

    records = []
    for method, times in timings.items():
        n_rows, rows_masked, n_chars = last_stats[method]
        runtime = statistics.median(times)
        records.append({
            "run_id": RUN_ID,
            "run_timestamp": RUN_TIMESTAMP,
            "sweep": sweep,
            "dataset": dataset,
            "method": method,
            "n_rows": int(n_rows),
            "n_cols": n_cols,
            "n_cells": int(n_rows) * n_cols,
            "input_mb": n_chars / 1e6,
            "rows_masked": int(rows_masked),
            "pct_rows_masked": 100 * rows_masked / n_rows if n_rows else 0.0,
            "runtime_s": runtime,
            "runtime_min_s": min(times),
            "runtime_max_s": max(times),
            "rows_per_s": n_rows / runtime,
            "cells_per_s": n_rows * n_cols / runtime,
            "mb_per_s": n_chars / 1e6 / runtime,
            "repeats": REPEATS,
            # Untimed scoring run, so it doesn't affect the runtime figures
            **(score_table(table, method, pcn_columns) or {}),
        })
    return records

In [0]:
bench_records = []

sweeps = {
    "rows": [(n, ROW_SWEEP_COLS) for n in ROW_SIZES],
    "columns": [(COL_SWEEP_ROWS, c) for c in COLUMN_COUNTS],
}
for sweep, grid in sweeps.items():
    for n_rows, n_cols in grid:
        print(f"[{sweep}] {n_rows:,} rows × {n_cols} cols")
        table = build_bench_table(n_rows, n_cols)
        bench_records += benchmark_table(table, f"synthetic_{n_rows}r_{n_cols}c", sweep)

for name in SHAPE_TABLES:
    table = f"{EVAL_DATABASE}.{name}"
    if not spark.catalog.tableExists(table):
        print(f"[real_tables] skipping {table} (not found)")
        continue
    print(f"[real_tables] {table}")
    bench_records += benchmark_table(table, name, "real_tables", PCN_COLUMNS.get(name))

bench_df = pd.DataFrame(bench_records)
print(f"\n✔ {len(bench_df)} benchmark results collected.")

### Results

`luhn_overhead_pct` is the extra runtime the Luhn check costs over the baseline (positive = slower).

In [0]:
RESULT_COLS = [
    "sweep", "dataset", "method", "n_rows", "n_cols", "input_mb", "pct_rows_masked",
    "runtime_s", "runtime_min_s", "runtime_max_s", "rows_per_s", "cells_per_s", "mb_per_s",
]
display(bench_df[RESULT_COLS].round(3))

overhead_df = bench_df.pivot_table(
    index=["sweep", "dataset", "n_rows", "n_cols"], columns="method", values="runtime_s"
).reset_index()
overhead_df["luhn_overhead_pct"] = 100 * (overhead_df["regex_luhn"] / overhead_df["regex_only"] - 1)
display(overhead_df.round(3))

In [0]:
def style_axes(ax, title, xlabel, ylabel):
    ax.set_title(title, loc="left", fontsize=11)
    ax.set_xlabel(xlabel, color="#52514e")
    ax.set_ylabel(ylabel, color="#52514e")
    ax.grid(axis="y", color="#e4e3df", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#c3c2b7")
    ax.tick_params(colors="#52514e")


def plot_sweep(ax, data, x, y):
    for method in METHODS:
        d = data[data["method"] == method].sort_values(x)
        linestyle, marker = METHOD_STYLES[method]
        ax.plot(
            d[x], d[y], color=METHOD_COLORS[method], linewidth=2, linestyle=linestyle,
            marker=marker, markersize=8, markeredgecolor="white", markeredgewidth=1.5,
            label=METHOD_LABELS[method],
        )
    ax.legend(frameon=False)


row_data = bench_df[bench_df["sweep"] == "rows"]
col_data = bench_df[bench_df["sweep"] == "columns"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

plot_sweep(axes[0, 0], row_data, "n_rows", "runtime_s")
axes[0, 0].set_xscale("log")
style_axes(axes[0, 0], f"Runtime vs rows ({ROW_SWEEP_COLS} columns)", "Rows (log scale)", "Median runtime (s)")

plot_sweep(axes[0, 1], row_data, "n_rows", "rows_per_s")
axes[0, 1].set_xscale("log")
style_axes(axes[0, 1], f"Throughput vs rows ({ROW_SWEEP_COLS} columns)", "Rows (log scale)", "Rows per second")

plot_sweep(axes[1, 0], col_data, "n_cols", "runtime_s")
style_axes(axes[1, 0], f"Runtime vs columns ({COL_SWEEP_ROWS:,} rows)", "Columns", "Median runtime (s)")

plot_sweep(axes[1, 1], col_data, "n_cols", "cells_per_s")
style_axes(axes[1, 1], f"Throughput vs columns ({COL_SWEEP_ROWS:,} rows)", "Columns", "Cells per second")

for ax in axes.flat:
    ax.set_ylim(bottom=0)
for ax in (axes[0, 1], axes[1, 1]):
    ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
plt.tight_layout()
plt.show()

### Accuracy at scale

Cell-level scores for every labelled dataset: a cell is a **TP** if it holds a PCN and was masked, **FP** if it was masked without holding one, **FN** if it holds a PCN left unmasked, **TN** otherwise. If accuracy holds at scale, each method's scores stay flat across dataset sizes and agree with Part 1.

`misaligned_rows` must be 0: it counts rows where masking damaged a `||` delimiter, which would shift values into the wrong columns in the pipeline.

In [0]:
ACC_METRICS = ["accuracy", "precision", "recall", "f1"]
ACC_COLS = [
    "sweep", "dataset", "method", "n_rows", "n_cols",
    "cell_tp", "cell_fp", "cell_fn", "cell_tn", *ACC_METRICS, "misaligned_rows",
]
scored_df = bench_df.dropna(subset=["accuracy"])
display(scored_df[ACC_COLS].round(4))

synthetic = scored_df[scored_df["sweep"].isin(["rows", "columns"])]
print("Spread of each metric across all synthetic dataset sizes (min / max):")
display(synthetic.groupby("method")[ACC_METRICS].agg(["min", "max"]).round(4))

print("Part 1 corpus results (field-level) for reference:")
display(pd.DataFrame({m: {k: metrics[m][f"field_{k}"] for k in ACC_METRICS} for m in METHODS}).T.round(4))

bad = scored_df[scored_df["misaligned_rows"] > 0]
if len(bad):
    print(f"✘ {len(bad)} run(s) with misaligned rows — delimiter integrity broken:")
    display(bad[ACC_COLS])
else:
    print("✔ No misaligned rows: masking never disturbed a '||' delimiter.")

In [0]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, metric in zip(axes, ["precision", "recall", "f1"]):
    plot_sweep(ax, scored_df[scored_df["sweep"] == "rows"], "n_rows", metric)
    ax.set_xscale("log")
    style_axes(ax, f"{metric.capitalize()} vs rows ({ROW_SWEEP_COLS} columns)", "Rows (log scale)", metric.capitalize())
    ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [0]:
(
    spark.createDataFrame(bench_df)
    .write.format("delta").mode("append").option("mergeSchema", "true")
    .saveAsTable(f"{EVAL_DATABASE}.{BENCHMARK_TABLE}")
)
print(f"✔ Benchmark results appended to '{EVAL_DATABASE}.{BENCHMARK_TABLE}'.")

---
## Part 4 — Resource Estimate

Linear extrapolation from the largest row-sweep run. Treat it as a first estimate for the resource worksheet: it assumes the same compute, column count and PCN density, and throughput that stays flat as data grows (check the throughput-vs-rows chart before relying on it).

In [0]:
TARGET_ROWS = 100_000_000

largest = row_data.sort_values("n_rows").groupby("method").tail(1)
for _, r in largest.iterrows():
    est_min = TARGET_ROWS / r["rows_per_s"] / 60
    print(
        f"{METHOD_LABELS[r['method']]:>22}: {r['rows_per_s']:,.0f} rows/s at {r['n_rows']:,} rows "
        f"→ ~{est_min:,.1f} min for {TARGET_ROWS:,} rows × {r['n_cols']} cols"
    )